# Post-processing and aggregation of detections

In [ ]:
from datetime import datetime
import json
import os
import requests

import geopandas as gpd
import shapely.geometry as sg

from zwerfafval_detectie.utils_eval import read_annotations_folder

RD_CRS = "EPSG:28992"  # CRS code for the Dutch Rijksdriehoek coordinate system
LAT_LON_CRS = "EPSG:4326"  # CRS code for WGS84 latitude/longitude coordinate system

In [ ]:
date = "260824"

predictions_folder = f"../datasets/experiments/zwerfafval/predict/inwinning_{date}_26m_v1e2"
output_folder = "../datasets/experiments/zwerfafval/heatmap"

city_geojson = "../datasets/experiments/zwerfafval/gebieden_v1_buurten_openbaar.geojson"

if date == "250514":
    metadata_file = f"../datasets/experiments/zwerfafval/data_inwinning_{date}/frames_gdf.gpkg"
elif date == "260421":
    metadata_file = f"../datasets/experiments/zwerfafval/data_inwinning_{date}/frames_1fps_gdf.gpkg"
else:
    metadata_json_folder = f"../datasets/experiments/zwerfafval/data_inwinning_{date}/json_tasks"

categories = {
    0: "Zwerfafval (grof)",
    1: "Zwerfafval (fijn)"
}

confidence = 0.3

os.makedirs(output_folder, exist_ok=True)

In [ ]:
# Load model predictions

predictions_gdf = read_annotations_folder(folder_path=predictions_folder, categories=categories)
predictions_gdf["file_name"] = predictions_gdf["file_name"].apply(lambda f: os.path.splitext(f)[0])

_predictions_sorted = (
    predictions_gdf[predictions_gdf["confidence"] >= confidence]
    .set_index("file_name")
    .sort_index()
)


# Count predictions per image

counts_df = (
    _predictions_sorted[["category"]]
    .replace(categories)
    .groupby(["file_name", "category"])
    .size()
    .unstack(fill_value=0)
)

In [ ]:
if date in {"250514", "260421"}:
    # Load metadata file (for 260421 or 250514)
    metadata_gdf = gpd.read_file(metadata_file)
    metadata_gdf["file_name"] = metadata_gdf["file_name"].apply(lambda f: os.path.splitext(f)[0])
    metadata_gdf.rename(columns={"frame_timestamp": "timestamp_utc"}, inplace=True)
    metadata_gdf = metadata_gdf[["file_name", "timestamp_utc", "geometry"]].set_index("file_name")
else:
    # Load metadata from JSON files

    data = {
        "file_name": [],
        "timestamp_utc": [],
        "geometry": [],
    }

    metadata_files = sorted([
        file for file in os.listdir(metadata_json_folder) 
        if os.path.splitext(file)[1] == ".json"
    ])

    for file in metadata_files:
        with open(os.path.join(metadata_json_folder, file), 'r') as fh:
            json_content = json.load(fh)
            data["file_name"].append(
                os.path.splitext(json_content["image_file_name"])[0]
            )
            data["timestamp_utc"].append(
                datetime.fromisoformat(json_content["image_file_timestamp"])
            )
            data["geometry"].append(
                sg.Point((
                    json_content["gps_data"]["longitude"],
                    json_content["gps_data"]["latitude"]
                ))
            )

    metadata_gdf = gpd.GeoDataFrame(
        data=data,
        crs=LAT_LON_CRS
    ).set_index("file_name")

In [ ]:
# Merge object counts and metadata

counts_merged = (
    gpd.GeoDataFrame(counts_df.join(metadata_gdf, how="outer"))
    .fillna(value={
        "Zwerfafval (fijn)": 0,
        "Zwerfafval (grof)": 0,
    })
    .to_crs(RD_CRS)
)

In [ ]:
# Discard detections outside of Amsterdam (GPS glitches)

amsterdam_shape = gpd.read_file(city_geojson).to_crs(RD_CRS).union_all()

counts_merged = counts_merged[counts_merged.within(amsterdam_shape)]

In [ ]:
# Add street and park info

street_url = "https://maps.amsterdam.nl/open_geodata/geojson_lnglat.php?KAARTLAAG=STRAATNAMEN&THEMA=straatnamen"
response = requests.get(street_url)
streets_gdf = gpd.GeoDataFrame.from_features(response.json()["features"], crs=LAT_LON_CRS)

parks_url = "https://maps.amsterdam.nl/open_geodata/geojson_lnglat.php?KAARTLAAG=PARKPLANTSOENGROEN&THEMA=stadsparken"
response = requests.get(parks_url)
parks_gdf = gpd.GeoDataFrame.from_features(response.json()["features"], crs=LAT_LON_CRS)

# Merge street
counts_merged = (
    counts_merged
    .sjoin_nearest(
        right=streets_gdf[["STT_NAAM", "geometry"]].to_crs(RD_CRS), 
        how="left", 
        distance_col="dist_to_street"
    )
    .drop(columns="index_right")
    .rename(columns={"STT_NAAM": "straat_naam"})
)

# Merge park
counts_merged = (
    counts_merged
    .sjoin_nearest(
        right=parks_gdf[["Naam", "geometry"]].to_crs(RD_CRS), 
        how="left", 
        distance_col="dist_to_park"
    )
    .drop(columns="index_right")
    .rename(columns={"Naam": "park_naam"})
)

In [ ]:
# Select image if distance to previous selected image is larger than a threshold

min_distance = 5.0

points = counts_merged["geometry"].to_crs(RD_CRS)

previous = points.iloc[0]

selection = [False]*len(points)
selection[0] = True

for i, point in enumerate(points.iloc[1:]):
    if previous.distance(point) >= min_distance:
        previous = point
        selection[i+1] = True

counts_merged["selected"] = False
counts_merged.loc[:, "selected"] = selection

In [ ]:
# Compute average counts for each selected image by averaging all upcoming
# images until the next selected 

# The assumption is that the camera looks forward
# so the upcoming images are a good representation for the current situation

counts_merged.reset_index(inplace=True)
idx_selected = counts_merged.index[counts_merged["selected"]].tolist()

for i in range(0, len(idx_selected) - 1):
    start_idx = idx_selected[i]
    end_idx = idx_selected[i+1] - 1
    section_mean_grof = counts_merged.loc[start_idx:end_idx, "Zwerfafval (grof)"].mean()
    section_mean_fijn = counts_merged.loc[start_idx:end_idx, "Zwerfafval (fijn)"].mean()
    counts_merged.loc[start_idx, "section_mean_grof"] = section_mean_grof
    counts_merged.loc[start_idx, "section_mean_fijn"] = section_mean_fijn

counts_merged.set_index("file_name", inplace=True)

# If the last row was "selected", it won't have a mean so we unselect it
counts_merged.loc[counts_merged.index[-1], "selected"] = False

In [ ]:
# Compute rolling average over section counts
# A window of 5 means averaging over 25m stretches (since each section is 5m)

counts_merged.loc[counts_merged["selected"], "rolling_avg_grof"] = (
    counts_merged.loc[counts_merged["selected"], "section_mean_grof"]
    .rolling(window=5, min_periods=1, center=True).mean()
)

counts_merged.loc[counts_merged["selected"], "rolling_avg_fijn"] = (
    counts_merged.loc[counts_merged["selected"], "section_mean_fijn"]
    .rolling(window=5, min_periods=1, center=True).mean()
)

In [ ]:
counts_merged.to_file(
    filename=os.path.join(output_folder, f"counts_{date}.gpkg"),
    driver="GPKG"
)

In [ ]:
# Plot results on a map

from xyzservices import TileProvider

rolling = True

ams_tile_provider = TileProvider(
    name="Topografie, standaard visualisatie (WM)",
    url="https://t1.data.amsterdam.nl/topo_wm/{z}/{x}/{y}.png",
    attribution="data.amsterdam.nl",
)

if rolling:
    plot_column = "rolling_avg_grof"
else:
    plot_column = "section_mean_grof"

map = (
    counts_merged[counts_merged["selected"]].dropna()
    .explore(
        column=plot_column,
        cmap="YlOrRd",
        style_kwds={
            "style_function": lambda x: {"radius": 2*x["properties"][plot_column]},
            "fillOpacity": 0.75,
            "weight": 2
        },
        legend=True,
        tiles=ams_tile_provider
    )
)

if rolling:
    map.save(os.path.join(output_folder, f"heatmap_inwinning_{date}_sampled_roll.html"))
else:
    map.save(os.path.join(output_folder, f"heatmap_inwinning_{date}_sampled.html"))